Task 5.1: Problem Definition

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("../data/final/nifty_features_with_regime_5min.csv", low_memory=False)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)

In [3]:
df.shape

(289973, 83)

In [4]:
df.head()

,timestamp,symbol_fut,date_fut,expiry_fut,open_fut,high_fut,low_fut,close_fut,ltp_fut,settle_price_fut,...,high_pe,low_pe,close_pe,open_interest_pe,contracts_pe,theta_ce,rho_ce,theta_pe,rho_pe,market_regime
0,2025-01-14,NIFTY,14-Jan-2025,2025-01-30,23248.0,23339.0,23198.45,23271.75,23280.0,23271.75,...,NaN,NaN,359.25,150.0,NaN,-5.157115,10.298329,-8.022101,-6.323211,NaN
1,2025-01-14,NIFTY,14-Jan-2025,2025-01-30,23248.0,23339.0,23198.45,23271.75,23280.0,23271.75,...,NaN,NaN,359.25,150.0,NaN,-17.998223,7.434743,-8.022101,-6.323211,1.0
2,2025-01-14,NIFTY,14-Jan-2025,2025-01-30,23248.0,23339.0,23198.45,23271.75,23280.0,23271.75,...,525.25,481.05,507.60,14475.0,188.0,NaN,NaN,-2.502252,-20.251634,NaN
3,2025-01-14,NIFTY,14-Jan-2025,2025-01-30,23248.0,23339.0,23198.45,23271.75,23280.0,23271.75,...,534.95,521.60,521.60,3675.0,2.0,NaN,NaN,-2.417843,-20.816369,NaN
4,2025-01-14,NIFTY,14-Jan-2025,2025-01-30,23248.0,23339.0,23198.45,23271.75,23280.0,23271.75,...,500.00,484.95,499.95,2700.0,20.0,NaN,NaN,-2.620097,-19.742016,NaN


In [5]:
# Initialize position
df["position"] = 0

# EMA crossovers
bull_cross = (
    (df["ema_5"] > df["ema_15"]) &
    (df["ema_5"].shift(1) <= df["ema_15"].shift(1))
)

bear_cross = (
    (df["ema_5"] < df["ema_15"]) &
    (df["ema_5"].shift(1) >= df["ema_15"].shift(1))
)

# Long entry (Regime +1)
df.loc[bull_cross & (df["market_regime"] == 1), "position"] = 1

# Short entry (Regime -1)
df.loc[bear_cross & (df["market_regime"] == -1), "position"] = -1

# Carry forward position until exit
df["position"] = df["position"].replace(0, np.nan).ffill().fillna(0)

In [6]:
df["position"].value_counts()

position
 1.0    253453
-1.0     35534
 0.0       986
Name: count, dtype: int64

In [7]:
# Exit long when EMA5 crosses below EMA15
exit_long = (
    (df["position"] == 1) &
    (df["ema_5"] < df["ema_15"])
)

# Exit short when EMA5 crosses above EMA15
exit_short = (
    (df["position"] == -1) &
    (df["ema_5"] > df["ema_15"])
)

df.loc[exit_long | exit_short, "position"] = 0

# Forward fill again
df["position"] = df["position"].replace(0, np.nan).ffill().fillna(0)

In [8]:
df["position"].value_counts()

position
 1.0    253453
-1.0     35534
 0.0       986
Name: count, dtype: int64

In [9]:
df["pos_change"] = df["position"].diff()

df["pos_change"].value_counts(dropna=False)

pos_change
 0.0    289911
 2.0        30
-2.0        30
 NaN         1
-1.0         1
Name: count, dtype: int64

In [10]:
trades = []

entry_idx = None
entry_price = None
direction = None

for i in range(1, len(df)):
    prev_pos = df["position"].iloc[i - 1]
    curr_pos = df["position"].iloc[i]

    # ENTRY
    if prev_pos == 0 and curr_pos != 0:
        entry_idx = i
        entry_price = df["open"].iloc[i]
        direction = curr_pos

    # EXIT OR FLIP
    elif prev_pos != 0 and curr_pos != prev_pos and entry_idx is not None:
        exit_price = df["open"].iloc[i]
        exit_idx = i

        pnl = (exit_price - entry_price) * direction

        trades.append({
            "entry_time": df["timestamp"].iloc[entry_idx],
            "exit_time": df["timestamp"].iloc[exit_idx],
            "direction": direction,
            "entry_price": entry_price,
            "exit_price": exit_price,
            "pnl": pnl,
            "holding_period": exit_idx - entry_idx
        })

        # handle flip
        if curr_pos != 0:
            entry_idx = i
            entry_price = df["open"].iloc[i]
            direction = curr_pos
        else:
            entry_idx = None

In [11]:
trade_df = pd.DataFrame(trades)
trade_df.shape

(60, 7)

In [12]:
trade_df["pnl"].describe()

count     54.000000
mean     -78.509259
std      187.617825
min     -440.700000
25%     -207.012500
50%      -92.150000
75%       62.525000
max      417.500000
Name: pnl, dtype: float64

In [13]:
trade_df.to_csv("../results/baseline_trades.csv", index=False)

In [14]:
trade_df["return"] = trade_df["pnl"] / trade_df["entry_price"]

In [15]:
#Train / Test Split (TIME-BASED)
split_idx = int(len(trade_df) * 0.7)

train_trades = trade_df.iloc[:split_idx]
test_trades  = trade_df.iloc[split_idx:]


In [16]:
#metric functions
def sharpe_ratio(returns):
    return np.mean(returns) / np.std(returns) * np.sqrt(252) if np.std(returns) != 0 else 0

def sortino_ratio(returns):
    downside = returns[returns < 0]
    return np.mean(returns) / np.std(downside) * np.sqrt(252) if len(downside) > 0 else 0

def max_drawdown(cum_returns):
    roll_max = cum_returns.cummax()
    drawdown = cum_returns / roll_max - 1
    return drawdown.min()

def backtest_metrics(df):
    cum_returns = (1 + df["return"]).cumprod()

    return {
        "Total Return": cum_returns.iloc[-1] - 1,
        "Sharpe": sharpe_ratio(df["return"]),
        "Sortino": sortino_ratio(df["return"]),
        "Max Drawdown": max_drawdown(cum_returns),
        "Win Rate": (df["pnl"] > 0).mean(),
        "Profit Factor": df[df["pnl"] > 0]["pnl"].sum() / abs(df[df["pnl"] < 0]["pnl"].sum()),
        "Avg Trade Duration": df["holding_period"].mean(),
        "Total Trades": len(df)
    }

In [17]:
#compute metrics
train_metrics = backtest_metrics(train_trades)
test_metrics  = backtest_metrics(test_trades)

pd.DataFrame([train_metrics, test_metrics], index=["Train", "Test"])

,Total Return,Sharpe,Sortino,Max Drawdown,Win Rate,Profit Factor,Avg Trade Duration,Total Trades
Train,-1.000000,-5.128897,-4.341068,-1.556407,0.309524,0.271559,3627.214286,42
Test,-0.976663,-0.696825,-1.461785,-1.069051,0.388889,0.744604,7541.500000,18


In [18]:
pd.DataFrame([train_metrics, test_metrics], index=["Train", "Test"]) \
  .to_csv("../results/baseline_backtest_metrics.csv")

# PART 5: MACHINE LEARNING

### PART 5.1 — PROBLEM DEFINITION

#### Objective
Build a binary classifier that predicts if a trade will be profitable BEFORE entry